# COSC v0.1 — Quickstart

Load the v0.1-preview release and produce one model-comparison-style plot.

**Requirements**: `pip install pandas pyarrow matplotlib` (or `pip install -e '.[examples]'` from the repo root).

**Data**: download [`cosc-v0.1-preview-NL000Q.tar.gz`](https://github.com/Cybis320/cosc-tools/releases/download/v0.1-preview/cosc-v0.1-preview-NL000Q.tar.gz) and extract it next to this notebook (`tar xzf cosc-v0.1-preview-NL000Q.tar.gz`).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path('cosc-v0.1/country=NL/station=NL000Q/date=2025-10-01')
assert BASE.exists(), f'extract the tarball first; expected {BASE}'

## 1. What's in the release

Five tables (six if you count `mask_polygons`). For most model-comparison work the two headline tables are `trajectory_point_observations/` (per-emission summary, the censored-survival input) and `detection_events/` (per-frame attribution, the time-resolved evidence).

In [ ]:
videos = pd.read_parquet(BASE / 'videos.parquet')
events = pd.read_parquet(BASE / 'detection_events.parquet')
obs    = pd.read_parquet(BASE / 'trajectory_point_observations.parquet')
traj   = pd.read_parquet(BASE / 'trajectories.parquet')

v = videos.iloc[0]
print(f"video_id           : {v.video_id}")
print(f"video duration     : {v.start_ts_utc} → {v.end_ts_utc}")
print(f"n_frames           : {v.n_frames:,}")
print(f"GT status          : {v.gt_status} ({v.gt_edit_count} edits)")
print(f"trajectory points  : {len(traj):,}")
print(f"observation rows   : {len(obs):,}")
print(f"detection events   : {len(events):,}")
print(f"events source mix  : {events.source.value_counts().to_dict()}")
print(f"censored (fov exit): {obs.censored_fov_exit.sum():,} / {len(obs):,} ({obs.censored_fov_exit.mean()*100:.1f}%)")
print(f"censored (vid end) : {obs.censored_video_end.sum():,} / {len(obs):,} ({obs.censored_video_end.mean()*100:.1f}%)")

## 2. Per-trajectory-point oldest-observed-age curve for one flight

Pick the highest-coverage flight in this video, then plot the **last observed contrail age** as a function of the emission moment along the flight's track. This is the empirical lower-bound on per-emission-point contrail lifetime that a model (CoCiP / RFM / custom) should produce a survival curve against. Rows with `censored_fov_exit = True` are shown lighter — their true lifetime is ≥ what's plotted.

In [ ]:
# Top flight by # of observed emission points
by_flight = (obs.groupby('flight_id')
                .agg(n_pts=('emit_ts_utc', 'size'),
                     callsign=('flight_id', 'first'))
                .sort_values('n_pts', ascending=False))
top_fid = by_flight.index[0]
tr_meta = traj[traj.flight_id == top_fid].iloc[0]
print(f'top flight: {tr_meta.callsign or tr_meta.flight_number or tr_meta.tail_number}  '
      f'(fid {top_fid[:8]}…, {by_flight.n_pts.iloc[0]} observed points)')

df = obs[obs.flight_id == top_fid].sort_values('emit_ts_utc').copy()
df['minutes_into_video'] = (df.emit_ts_utc - videos.start_ts_utc.iloc[0]).dt.total_seconds() / 60.0

fig, ax = plt.subplots(figsize=(11, 4.5))
uncens = df[~df.censored_fov_exit]
cens   = df[df.censored_fov_exit]
ax.scatter(uncens.minutes_into_video, uncens.detect_last_age_s/60, s=18, color='steelblue', label='observed dissipation (uncensored)')
ax.scatter(cens.minutes_into_video,   cens.detect_last_age_s/60,  s=18, color='lightgrey', label='right-censored (FOV exit; lifetime ≥)')
ax.set_xlabel('emission time (min from video start)')
ax.set_ylabel('last observed contrail age (min)')
ax.set_title(f'COSC v0.1 — last observed contrail age vs emission time\nflight {tr_meta.callsign}, station NL000Q, 2025-10-01')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()

## 3. Where to go from here

- **Model comparison**: feed each row of `obs` (emit_ts/lat/lon/alt) through your survival model, compare predicted dissipation to `detect_last_age_s`, treat the censored rows as right-censored in your loss / KM estimator.
- **Time-resolved comparison**: use `events.age_s` to compare opacity/visibility curves at sub-row resolution.
- **Cross-station aggregation** (v0.2+): `pd.read_parquet('cosc-v0.1', engine='pyarrow').` All tables are partitioned `country=*/station=*/date=*`.
- **Schema details**: [`specs/SCHEMA.md`](https://github.com/Cybis320/cosc-tools/blob/main/specs/SCHEMA.md). Conventions, censoring semantics, and provenance fields all documented column-by-column.